# Week 10: MLOps, System Evaluation & Explainable AI
## Notebook 10: Algorithmic Fairness: Demographic Parity & Disparate Impact

### 1. Conceptual Intuition & Theoretical Background
Machine learning models trained on historical data can perpetuate or amplify demographic biases against protected attributes $A \in \{0, 1\}$ (e.g. race, gender, age).

**1. Demographic Parity (Statistical Parity):**
Acceptance rates must be equal across protected groups:
$$P(\hat{Y} = 1 \mid A = 0) = P(\hat{Y} = 1 \mid A = 1)$$
- **Disparate Impact Ratio (Four-Fifths Rule, EEOC):**
$$\text{Disparate Impact} = \frac{P(\hat{Y} = 1 \mid A = 0)}{P(\hat{Y} = 1 \mid A = 1)} \ge 0.80$$

**2. Equalized Odds (Hardt et al., 2016):**
Predictor $\hat{Y}$ satisfies Equalized Odds if true positive rate (TPR) and false positive rate (FPR) are identical across groups:
$$P(\hat{Y} = 1 \mid A = 0, Y = y) = P(\hat{Y} = 1 \mid A = 1, Y = y), \quad \forall y \in \{0, 1\}$$

---

### 2. Scratch Python Implementation: Algorithmic Fairness Audit

In [ ]:
import numpy as np

def compute_fairness_audit(y_true, y_pred, sensitive_attr):
    """
    Computes Disparate Impact, Demographic Parity, and Equalized Odds metrics.
    sensitive_attr: 0 for protected group, 1 for privileged group.
    """
    # Group masks
    prot_mask = (sensitive_attr == 0)
    priv_mask = (sensitive_attr == 1)
    
    # 1. Selection Rates (Positive Prediction Rates)
    selection_rate_prot = np.mean(y_pred[prot_mask])
    selection_rate_priv = np.mean(y_pred[priv_mask])
    
    disparate_impact_ratio = selection_rate_prot / max(1e-6, selection_rate_priv)
    statistical_parity_diff = selection_rate_prot - selection_rate_priv
    
    # 2. Equalized Odds: True Positive Rate & False Positive Rate
    tpr_prot = np.sum((y_pred == 1) & (y_true == 1) & prot_mask) / np.sum((y_true == 1) & prot_mask)
    tpr_priv = np.sum((y_pred == 1) & (y_true == 1) & priv_mask) / np.sum((y_true == 1) & priv_mask)
    
    fpr_prot = np.sum((y_pred == 1) & (y_true == 0) & prot_mask) / np.sum((y_true == 0) & prot_mask)
    fpr_priv = np.sum((y_pred == 1) & (y_true == 0) & priv_mask) / np.sum((y_true == 0) & priv_mask)
    
    return {
        'Selection_Rate_Protected': selection_rate_prot,
        'Selection_Rate_Privileged': selection_rate_priv,
        'Disparate_Impact_Ratio': disparate_impact_ratio,
        'Statistical_Parity_Difference': statistical_parity_diff,
        'TPR_Difference': abs(tpr_prot - tpr_priv),
        'FPR_Difference': abs(fpr_prot - fpr_priv),
        'Passes_80_Percent_Rule': disparate_impact_ratio >= 0.80
    }

# Synthetic Hiring Model Simulation
np.random.seed(42)
n_candidates = 1000
sensitive_group = np.random.binomial(1, 0.6, n_candidates) # 60% privileged

# Ground truth qualified status (equal true capability)
true_qualified = np.random.binomial(1, 0.5, n_candidates)

# Model predictions contaminated with historical screening bias
y_predicted = true_qualified.copy()
# Reduce positive selections for protected group
bias_mask = (sensitive_group == 0) & (np.random.rand(n_candidates) < 0.35)
y_predicted[bias_mask] = 0

audit = compute_fairness_audit(true_qualified, y_predicted, sensitive_group)

print("--- Algorithmic Fairness Audit Report ---")
print(f"Selection Rate (Privileged Group):  {audit['Selection_Rate_Privileged']*100:.1f}%")
print(f"Selection Rate (Protected Group):   {audit['Selection_Rate_Protected']*100:.1f}%")
print(f"Disparate Impact Ratio:             {audit['Disparate_Impact_Ratio']:.3f}")
print(f"EEOC 80% Rule Compliance:          {audit['Passes_80_Percent_Rule']} (VIOLATION: < 0.80)")
print(f"Equalized Odds TPR Discrepancy:     {audit['TPR_Difference']:.4f}")

### 3. Step-by-Step Code Walkthrough
1. **Selection Rate:** Computes favorable outcome probabilities per demographic group.
2. **Disparate Impact Ratio:** Flags systematic underrepresentation below 80%.
3. **Equal Opportunity Metric:** Evaluates TPR differences to check whether qualified candidates are treated equally.

---

### 4. Industry Applications & Trade-offs
- **Credit Approval & Hiring Screening (EU AI Act):** Mandatory bias auditing before deploying automated risk tools.
- **Trade-off:** Impossibility Theorem of Fairness (Kleinberg et al.): Demographic Parity, Equalized Odds, and Calibration cannot be satisfied simultaneously unless base rates are identical.

---

### 5. Top Interview Questions & Answers
**Q1: What is the "Four-Fifths Rule" (80% Rule) in disparate impact law?**
> **Answer:** A selection rate for any group that is less than four-fifths (80%) of the rate for the group with the highest selection rate is considered prima facie evidence of adverse disparate impact by regulatory bodies (e.g. EEOC).

**Q2: Can dropping the sensitive attribute (e.g. race or gender) prevent algorithmic bias?**
> **Answer:** No ("Fairness through Unawareness" fallacy). Proxy features (e.g. zip codes, education history, browsing patterns) allow models to reconstruct sensitive attributes implicitly.